# Long COVID: Neuropsychiatric Symptoms — Global Data (2021–2024)

**DOI:** [![DOI](https://img.shields.io/badge/DOI-10.7910%2FDVN%2F7LRXMV-blue)](https://doi.org/10.7910/DVN/7LRXMV)  
**Author:** Juan Moises de la Serna | ORCID: [0000-0002-8401-8018](https://orcid.org/0000-0002-8401-8018)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmoisesd/long-covid-neuropsychiatric-symptoms-global-2021-2024/blob/main/notebooks/long_covid_neuropsychiatric_analysis.ipynb)

## Overview
Analysis of neuropsychiatric manifestations in Long COVID patients: brain fog, depression, anxiety, and cognitive impairment.

**Key statistics:**
- **Brain fog**: 22–32% of Long COVID patients
- **Depression**: 23–26% prevalence in Long COVID
- **Anxiety**: 22–24% prevalence
- **Cognitive impairment**: 20–30% (up to 3x normal)
- Symptoms persist 3–12+ months post-infection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
print('✓ Libraries loaded')

In [ ]:
# ─── Simulated representative dataset ─────────────────────────────────────────
np.random.seed(2024)
n = 500
status = np.random.choice(['Long_COVID','Recovered','Control'], n, p=[0.35,0.35,0.30])

df = pd.DataFrame({
    'subject_id': [f'LC{i:04d}' for i in range(1, n+1)],
    'status': status,
    'age': np.random.randint(18, 72, n),
    'sex': np.random.choice(['F','M'], n, p=[0.58,0.42]),
    'country': np.random.choice(['USA','UK','Spain','Germany','Brazil','Italy'], n),
    'months_post_covid': np.where(status=='Control', 0,
                         np.random.choice(range(1, 25), n)),
    'brain_fog': np.where(status=='Long_COVID', np.random.binomial(1,0.27,n),
                  np.where(status=='Recovered', np.random.binomial(1,0.09,n), np.random.binomial(1,0.05,n))),
    'depression_PHQ9': np.where(status=='Long_COVID', np.random.normal(10.8,4.2,n),
                        np.where(status=='Recovered', np.random.normal(5.1,3.3,n), np.random.normal(3.2,2.8,n))).clip(min=0,max=27),
    'anxiety_GAD7': np.where(status=='Long_COVID', np.random.normal(9.3,4.0,n),
                     np.where(status=='Recovered', np.random.normal(4.5,3.1,n), np.random.normal(2.9,2.5,n))).clip(min=0,max=21),
    'cognitive_score': np.where(status=='Long_COVID', np.random.normal(62,12,n),
                        np.where(status=='Recovered', np.random.normal(78,9,n), np.random.normal(85,8,n))).clip(min=0,max=100),
    'fatigue_score': np.where(status=='Long_COVID', np.random.normal(7.1,2.0,n),
                      np.where(status=='Recovered', np.random.normal(3.9,2.1,n), np.random.normal(2.1,1.6,n))).clip(min=0,max=10)
})
df['depression_PHQ9'] = df['depression_PHQ9'].round(1)
df['anxiety_GAD7'] = df['anxiety_GAD7'].round(1)
df['cognitive_score'] = df['cognitive_score'].round(1)

print(f'Dataset shape: {df.shape}')
print(f'Status distribution:\n{df.status.value_counts()}')
df.head()

In [ ]:
# ─── Figure 1: Symptom Comparison Across Groups ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['depression_PHQ9', 'anxiety_GAD7', 'cognitive_score', 'fatigue_score']
titles  = ['Depression (PHQ-9)', 'Anxiety (GAD-7)', 'Cognitive Score', 'Fatigue Score']
palette = {'Long_COVID': '#E74C3C', 'Recovered': '#F39C12', 'Control': '#27AE60'}
order = ['Control', 'Recovered', 'Long_COVID']

for ax, metric, title in zip(axes.flat, metrics, titles):
    sns.violinplot(data=df, x='status', y=metric, order=order, palette=palette, ax=ax, cut=0, linewidth=1)
    sns.stripplot(data=df, x='status', y=metric, order=order, palette=palette, ax=ax, size=1.5, alpha=0.25)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('')
    ax.set_xticklabels(['Control','Recovered','Long COVID'], fontsize=9)

plt.suptitle('Neuropsychiatric Symptoms: Long COVID vs Controls\nDOI: 10.7910/DVN/7LRXMV',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_symptom_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figure 2: Symptoms over time post-COVID ─────────────────────────────────
long_covid_df = df[df['status']=='Long_COVID'].copy()
long_covid_df['month_bin'] = pd.cut(long_covid_df['months_post_covid'],
    bins=[0,3,6,9,12,24], labels=['1-3m','4-6m','7-9m','10-12m','>12m'])

fig, ax = plt.subplots(figsize=(11, 6))
for metric, color, marker in [
    ('depression_PHQ9','#E74C3C','o'), ('anxiety_GAD7','#F39C12','s'),
    ('fatigue_score','#9B59B6','^')]:
    gb = long_covid_df.groupby('month_bin', observed=True)[metric].agg(['mean','sem'])
    ax.errorbar(range(len(gb)), gb['mean'], yerr=gb['sem'], fmt=f'-{marker}',
                color=color, lw=2, markersize=8, capsize=4, label=metric.replace('_',' '))

ax.set_xticks(range(5))
ax.set_xticklabels(['1-3m','4-6m','7-9m','10-12m','>12m'], fontsize=11)
ax.set_xlabel('Months post-COVID-19 infection', fontsize=12)
ax.set_ylabel('Mean Score', fontsize=12)
ax.set_title('Neuropsychiatric Symptom Trajectory Over Time\nDOI: 10.7910/DVN/7LRXMV', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig2_symptom_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figure 3: Prevalence by Country ──────────────────────────────────────────
prevalence_data = {
    'Country': ['USA','UK','Spain','Germany','Brazil','Italy','China','France','India','Australia'],
    'brain_fog_pct': [28.3,31.1,24.5,22.8,26.9,29.4,19.2,23.6,21.4,25.7],
    'depression_pct': [25.1,26.8,22.4,20.9,28.3,24.7,17.5,21.8,23.1,22.3],
    'anxiety_pct': [23.4,24.1,21.8,19.6,26.5,23.2,16.8,20.4,22.0,21.7]
}
df_prev = pd.DataFrame(prevalence_data)

x = np.arange(len(df_prev))
width = 0.28
fig, ax = plt.subplots(figsize=(14,6))
bars1 = ax.bar(x - width, df_prev['brain_fog_pct'], width, label='Brain Fog', color='#E74C3C', alpha=0.85)
bars2 = ax.bar(x, df_prev['depression_pct'], width, label='Depression', color='#3498DB', alpha=0.85)
bars3 = ax.bar(x + width, df_prev['anxiety_pct'], width, label='Anxiety', color='#27AE60', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_prev['Country'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Prevalence (%)', fontsize=12)
ax.set_title('Long COVID Neuropsychiatric Symptom Prevalence by Country\nDOI: 10.7910/DVN/7LRXMV', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig3_prevalence_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Statistical Comparison ───────────────────────────────────────────────────
from scipy.stats import ttest_ind
print('=== Long COVID vs Control — Statistical Tests ===')
for metric in ['depression_PHQ9','anxiety_GAD7','cognitive_score','fatigue_score']:
    lc = df[df.status=='Long_COVID'][metric]
    ct = df[df.status=='Control'][metric]
    t, p = ttest_ind(lc, ct)
    d = (lc.mean()-ct.mean()) / np.sqrt((lc.std()**2+ct.std()**2)/2)
    print(f'{metric:20s}: LC mean={lc.mean():.2f}, HC mean={ct.mean():.2f}, '
          f't={t:.2f}, p={p:.3e}, Cohen\'d={d:.2f}')

## Citation
```bibtex
@data{DVN/7LRXMV_2024,
  author    = {de la Serna, Juan Moises},
  title     = {Long COVID Neuropsychiatric Symptoms: Global Prevalence and Longitudinal Data 2021-2024},
  year      = {2024},
  publisher = {Harvard Dataverse},
  doi       = {10.7910/DVN/7LRXMV},
  url       = {https://doi.org/10.7910/DVN/7LRXMV}
}
```
## License: CC0 1.0 Universal